In [1]:
import os
import pickle
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay, accuracy_score, roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from xgboost import XGBClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from imblearn.over_sampling import SMOTE
from hmmlearn import hmm
from hmmlearn.hmm import GaussianHMM
import plotly.express as px
import mlflow
from mlflow.models.signature import infer_signature
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
files = ['two_class_raw_1s_no.csv', 'two_class_raw_1s_yo_0.5.csv', 'two_class_raw_1s_yo_0.8.csv', 'two_class_raw_2s_no.csv', 'two_class_raw_2s_yo_0.5.csv', 'two_class_raw_2s_yo_0.8.csv', 
        'two_class_raw_3s_no.csv', 'two_class_raw_3s_yo_0.5.csv', 'two_class_raw_3s_yo_0.8.csv', 'two_class_raw_4s_no.csv', 'two_class_raw_4s_yo_0.5.csv', 'two_class_raw_4s_yo_0.8.csv',
        'two_class_raw_5s_no.csv', 'two_class_raw_5s_yo_0.5.csv', 'two_class_raw_5s_yo_0.8.csv']

base_path = '/home/edumaba/Public/MPhil_Thesis/Code/uropatch-data-analysis/features_data_main/'

In [3]:
samp_tech= ['os', 'us', 'beide','none']

In [ ]:
def setup_xgboost_with_imbalance_handling(X_train, y_train, samp_tech='os', original_scale_weight=1.27):
    """
    Set up XGBoost with appropriate imbalance handling
    """
    if samp_tech == 'os':
        print("Applying SMOTE...")
        smote = SMOTE(random_state=42)
        X_balanced, y_balanced = smote.fit_resample(X_train, y_train)
        
        # Check if we still need scale_pos_weight
        unique, counts = np.unique(y_balanced, return_counts=True)
        ratio = counts[0] / counts[1]
        
        if abs(ratio - 1.0) > 0.1:
            scale_weight = ratio
            print(f"Post-SMOTE imbalance detected, using scale_pos_weight: {scale_weight:.3f}")
        else:
            scale_weight = None
            print("Classes balanced after SMOTE, not using scale_pos_weight")
    elif samp_tech == "us":
        pass
    
    elif samp_tech =='beide':
        pass
    else:
        print("No SMOTE applied, using original scale_pos_weight")
        X_balanced, y_balanced = X_train, y_train
        scale_weight = original_scale_weight
    
    # Configure XGBoost
    xgb_params = {        
        # 'n_estimators':200,
        # 'max_depth':5,
        # 'learning_rate':0.1,
        # 'subsample':0.8,
        # 'colsample_bytree':0.8,
        'random_state':42,
        'eval_metric':'logloss',
        # 'reg_alpha':0.1,   # L1 regularization
        # 'reg_lambda':1.0,  # L2 regularization
        # 'verbosity':0
    }
    
    if scale_weight is not None:
        xgb_params['scale_pos_weight'] = scale_weight
        
        
        

    
    model = XGBClassifier(
        random_state=42,
        eval_metric='logloss',  # Suppress warning
        nthread= 1,  # Force single thread
        tree_method= 'exact'  # Use exact tree method
        
    )
    
    return model, X_balanced, y_balanced, xgb_params

In [5]:
# smote = SMOTE(random_state=42)
# X_balanced, y_balanced = smote.fit_resample(X_train, y_train)
# print("Before SMOTE")
# print("==================")
# unique, counts = np.unique(y_train, return_counts=True)
# print(f"{unique[0]}: {counts[0]}")
# print(f"{unique[1]}: {counts[1]}")
# print("After SMOTE")
# print("==================")
# bal_unique, bal_counts = np.unique(y_balanced, return_counts=True)
# print(f"{bal_unique[0]}: {bal_counts[0]}")
# print(f"{bal_unique[1]}: {bal_counts[1]}")

In [ ]:
results = {}
for file in files:
    data_path = os.path.join(base_path, file)
    features = pd.read_csv(data_path)
    features.drop(['center_time', 'start_time', 'end_time'], axis=1, inplace=True)
    details = file.split('_')
    exp_name = f"{details[3]}_{details[-1].replace('.csv', '')}"
    print(f"Analysing {exp_name}")
    
    # split data
    X = features.drop(columns=['label', 'experiment_id'])
    y = features['label']
    groups = features['experiment_id']

    splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    train_idx, test_idx = next(splitter.split(X, y, groups))

    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    groups_train, groups_test = groups.iloc[train_idx], groups.iloc[test_idx]
    
    
    
    # cross-validation
    gkf = GroupKFold(n_splits=10)
    
    
    fold_results = []

    for fold_num, (train_idx_fold, test_idx_fold) in enumerate(gkf.split(X_train, y_train, groups_train)):
        X_train_fold, X_test_fold = X_train.iloc[train_idx_fold], X_train.iloc[test_idx_fold]
        y_train_fold, y_test_fold = y_train.iloc[train_idx_fold], y_train.iloc[test_idx_fold]
        
        
        # Apply sampling within each fold
        model, X_train_fold_sampled, y_train_fold_sampled, xgb_params = setup_xgboost_with_imbalance_handling(
            X_train_fold, 
            y_train_fold,
            samp_tech='os',  # 'os' applies smote to the minority class
            original_scale_weight=1.27
        )
        
        # Encode labels
        label_encoder = LabelEncoder()
        y_train_encoded = label_encoder.fit_transform(y_train_fold_sampled)
        y_test_encoded = label_encoder.transform(y_test_fold)
        
        
        model.fit(X_train_fold_sampled, y_train_encoded)
        
        # Evaluate
        predictions = label_encoder.inverse_transform(model.predict(X_test_fold))
        accuracy = accuracy_score(y_test_fold, predictions)
        report_dict = classification_report(y_test_fold, predictions, output_dict=True)
        
        
        precision_void = report_dict.get("void", {}).get("precision", 0.0)
        recall_void = report_dict.get("void", {}).get("recall", 0.0)
        f1_void = report_dict.get("void", {}).get("f1-score", 0.0)

        precision_non_void = report_dict.get("non-void", {}).get("precision", 0.0)
        recall_non_void = report_dict.get("non-void", {}).get("recall", 0.0)
        f1_non_void = report_dict.get("non-void", {}).get("f1-score", 0.0)
        
        macro_f1 = report_dict.get("macro avg", {}).get("f1-score", 0.0)
        weighted_f1 = report_dict.get("weighted avg", {}).get("f1-score", 0.0)
        
        # Store results
        fold_results.append({
            'fold': fold_num,
            'recall_void': recall_void,
            'precision_void': precision_void,
            'f1_void': f1_void,
            'macro_f1': macro_f1,
            'accuracy': accuracy,
            'precision_non_void': precision_non_void,
            'recall_non_void': recall_non_void,
            'f1_non_void': f1_non_void, 
            'weighted_f1': weighted_f1
            # 'train_idx': train_idx_fold,
            # 'test_idx': test_idx_fold,
            # 'train_groups': groups_train.iloc[train_idx_fold],
            # 'test_groups': groups_train.iloc[test_idx_fold]
        })
        
    results[exp_name]= fold_results
    print("-" * 50)

        # print(f"Fold {fold_num} Accuracy: {accuracy * 100:.2f}%")

    # # Find best performing fold
    # best_fold = max(fold_results, key=lambda x: x['accuracy'])
    # # print(f"\nBest fold: {best_fold['fold']} with accuracy: {best_fold['accuracy'] * 100:.2f}%")

    # fs_accuracy_cv = round(np.mean([f['accuracy'] for f in fold_results]) * 100, 2)
    # fs_std_cv = round(np.std([f['accuracy'] for f in fold_results]) * 100, 2)
    # fs_var_cv = round(np.var([f['accuracy'] for f in fold_results]) * 100, 2)

    # print(f"Mean accuracy across all folds: {fs_accuracy_cv}%")
    # print(f"Standard deviation of accuracy across folds: {fs_std_cv}%")
    # print(f"Variance of accuracy across folds: {fs_var_cv}%")
    
    

Analysing 1s_no
Applying SMOTE...
Classes balanced after SMOTE, not using scale_pos_weight
Applying SMOTE...
Classes balanced after SMOTE, not using scale_pos_weight
Applying SMOTE...
Classes balanced after SMOTE, not using scale_pos_weight
Applying SMOTE...
Classes balanced after SMOTE, not using scale_pos_weight
Applying SMOTE...
Classes balanced after SMOTE, not using scale_pos_weight
Applying SMOTE...
Classes balanced after SMOTE, not using scale_pos_weight
Applying SMOTE...
Classes balanced after SMOTE, not using scale_pos_weight
Applying SMOTE...
Classes balanced after SMOTE, not using scale_pos_weight
Applying SMOTE...
Classes balanced after SMOTE, not using scale_pos_weight
Applying SMOTE...
Classes balanced after SMOTE, not using scale_pos_weight
--------------------------------------------------


In [7]:
df = pd.DataFrame(results)
df.head()

,1s_no
0,"{'fold': 0, 'recall_void': 0.4528301886792453,..."
1,"{'fold': 1, 'recall_void': 0.3191489361702128,..."
2,"{'fold': 2, 'recall_void': 0.5303030303030303,..."
3,"{'fold': 3, 'recall_void': 0.45652173913043476..."
4,"{'fold': 4, 'recall_void': 0.38461538461538464..."


In [8]:
df_1 = pd.DataFrame(results['1s_no'])
df_1

,fold,recall_void,precision_void,f1_void,macro_f1,accuracy,precision_non_void,recall_non_void,f1_non_void,class_distribution_train,class_distribution_test
0,0,0.452830,0.648649,0.533333,0.683333,0.754386,0.783582,0.889831,0.833333,"{'non-void': 932, 'void': 932}","{'non-void': 118, 'void': 53}"
1,1,0.319149,0.340909,0.329670,0.547977,0.653409,0.757576,0.775194,0.766284,"{'non-void': 921, 'void': 921}","{'non-void': 129, 'void': 47}"
2,2,0.530303,0.479452,0.503597,0.589827,0.607955,0.699029,0.654545,0.676056,"{'non-void': 940, 'void': 940}","{'non-void': 110, 'void': 66}"
3,3,0.456522,0.308824,0.368421,0.520574,0.568862,0.747475,0.611570,0.672727,"{'non-void': 929, 'void': 929}","{'non-void': 121, 'void': 46}"
4,4,0.384615,0.500000,0.434783,0.516360,0.530120,0.547170,0.659091,0.597938,"{'non-void': 962, 'void': 962}","{'non-void': 88, 'void': 78}"
5,5,0.511364,0.714286,0.596026,0.669321,0.685567,0.671756,0.830189,0.742616,"{'non-void': 944, 'void': 944}","{'non-void': 106, 'void': 88}"
6,6,0.619048,0.597701,0.608187,0.648280,0.652850,0.698113,0.678899,0.688372,"{'non-void': 941, 'void': 941}","{'non-void': 109, 'void': 84}"
7,7,0.634146,0.753623,0.688742,0.713086,0.715152,0.687500,0.795181,0.737430,"{'non-void': 967, 'void': 967}","{'non-void': 83, 'void': 82}"
8,8,0.430000,0.661538,0.521212,0.518275,0.518293,0.424242,0.656250,0.515337,"{'non-void': 986, 'void': 986}","{'void': 100, 'non-void': 64}"
9,9,0.592105,0.576923,0.584416,0.659976,0.676768,0.741667,0.729508,0.735537,"{'non-void': 928, 'void': 928}","{'non-void': 122, 'void': 76}"


In [9]:
# df_1_05 = pd.DataFrame(results['1s_0.5'])
# df_1_05